# 전략 실험 템플릿

새 전략을 노트북에서 실험한 뒤 패키지로 승격하기 위한 스크래치 패드입니다. 안정된 로직은 `.ipynb`에 두지 말고 `src/backtest/strategies/`로 옮기세요.

**커널:** 프로젝트 `.venv`.

**데이터 전제:** `data/market-data/curated/daily_prices/`가 있어야 합니다. 없으면 `uv run market-data backfill` 또는 `--data-dir data/market-data/test`.

두 경로 중 하나만 실행하면 됩니다.

- **경로 A:** 롱 온리 전량 매수/매도 + 다음 봉 시가 체결 → `golden_cross`/`death_cross` 컬럼 + `run_bar_by_bar`
- **경로 B:** 분할매수, 지정가, 상태머신 등 공유 엔진에 안 맞으면 커스텀 루프 → `BacktestResult`. 무한매수법은 이 경로로 승격할 예정입니다.

투자 조언을 제공하지 않습니다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd

from backtest.analytics import plot_backtest, summarize
from backtest.core import BacktestResult, run_bar_by_bar
from backtest.data import load_price_series

security_id = "KRX:005930"
data_dir = Path("data/market-data")
start = date(2015, 1, 1)
end = None
initial_cash = 10_000_000.0
fee_rate = 0.0015

prices = load_price_series(data_dir, security_id, start=start, end=end)
prices.head()

## 경로 A — 공유 엔진

`prepare()`가 `golden_cross`(매수 신호)와 `death_cross`(매도 신호)를 만들면 `run_bar_by_bar`가 다음 봉 시가에 전량 체결합니다. 아래는 SMA 크로스를 인라인으로 재현한 예입니다. 실험할 때는 이 셀의 시그널 로직만 바꾸세요.

In [ ]:
from backtest.strategies import add_sma_cross_signals

signaled = add_sma_cross_signals(prices, fast=50, slow=200)
# 커스텀 시그널이면 대신:
# signaled = prices.copy()
# signaled["golden_cross"] = ...
# signaled["death_cross"] = ...

result_a = run_bar_by_bar(signaled, initial_cash=initial_cash, fee_rate=fee_rate)
display(summarize(result_a))
result_a.trades

In [ ]:
plot_backtest(
    result_a,
    security_id=security_id,
    strategy="scratch_path_a",
    fast=50,
    slow=200,
    show=True,
)

## 경로 B — 커스텀 실행

공유 엔진을 쓰지 않을 때는 `BacktestResult`만 맞추면 `summarize` / `plot_backtest`를 재사용할 수 있습니다.

**equity 컬럼:** `trade_date`, `cash`, `shares`, `close`, `equity`, `position`  
**trades 컬럼:** `trade_date`, `side` (`buy`/`sell`), `price`, `shares`, `fee`, `cash_after`

아래는 첫 봉 시가에 전량 매수하고 끝까지 보유하는 최소 루프입니다. 실제 전략 상태로 바꾸세요.

In [ ]:
cash = float(initial_cash)
shares = 0.0
equity_rows: list[dict] = []
trade_rows: list[dict] = []

for i, row in prices.iterrows():
    open_price = float(row["open"])
    close_price = float(row["close"])
    trade_date = row["trade_date"]

    if shares == 0 and open_price > 0:
        cost_per_share = open_price * (1.0 + fee_rate)
        bought = cash / cost_per_share
        if bought > 0:
            fee = bought * open_price * fee_rate
            cash -= bought * cost_per_share
            shares = bought
            trade_rows.append(
                {
                    "trade_date": trade_date,
                    "side": "buy",
                    "price": open_price,
                    "shares": shares,
                    "fee": fee,
                    "cash_after": cash,
                }
            )

    equity = cash + shares * close_price
    equity_rows.append(
        {
            "trade_date": trade_date,
            "cash": cash,
            "shares": shares,
            "close": close_price,
            "equity": equity,
            "position": 1 if shares > 0 else 0,
        }
    )

result_b = BacktestResult(
    equity=pd.DataFrame(equity_rows),
    trades=pd.DataFrame(trade_rows),
    initial_cash=initial_cash,
    fee_rate=fee_rate,
)
display(summarize(result_b))
result_b.trades

In [ ]:
plot_backtest(
    result_b,
    security_id=security_id,
    strategy="scratch_path_b",
    fast=None,
    slow=None,
    show=True,
)

## 승격 체크리스트

로직이 안정되면:

1. `src/backtest/strategies/<name>.py`에 `Strategy` 구현 (`name`, 경로 A면 `prepare()` + `run_bar_by_bar`, 경로 B면 커스텀 `run()`)
2. `src/backtest/strategies/__init__.py`의 `STRATEGIES`에 등록
3. synthetic DataFrame 테스트 추가 (`tests/test_backtest_golden_cross.py` 패턴)
4. 이 노트북은 `run_backtest("<name>", ...)` 러너로 축소

상세는 [노트북 워크벤치](/docs/notebooks.md)를 보세요.